# InferCNV

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import matplotlib.pyplot as plt
import warnings
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['figure.figsize'] = (5, 5)

In [ ]:
adata = sc.read_h5ad("14_gene_expression.h5ad")

In [ ]:
adata

In [ ]:
adata.var

#### Prepare gene annotations

In [ ]:
genes_loc = pd.read_csv("data/genes_gtf/genes.v38.annotation.tab")
genes_loc

In [ ]:
#find dupicated gene names and make them unique
genes_loc['gene_name'] = genes_loc['gene_name'] + genes_loc.groupby("gene_name").cumcount().astype(str).radd("-").replace("-0",'')
genes_loc


In [ ]:
#check that it did the correct thing
genes_loc[genes_loc.gene_name.str.contains("TBCE")]

In [ ]:
#set gene names as index
genes_loc = genes_loc.set_index("gene_name")

In [ ]:
genes_loc = genes_loc.reindex(adata.var_names)

In [ ]:
genes_loc

In [ ]:
adata.var = pd.concat([adata.var, genes_loc], axis=1)
adata.var

In [ ]:
adata.var['chromosome'].isna().sum()

In [ ]:
adata.obs["new_celltype"].value_counts()

### MISC UMAP generation

In [ ]:
#sc.pl.umap(adata, color="new_celltype")

In [ ]:
#sc.pl.umap(adata, color="outcome_C12D29", palette=["orange", "blue"])

In [ ]:
#sc.pl.umap(adata, color="patient_alias")

### inferCNV with healthy patients included as control

In [ ]:
adata.obs['leiden_and_celltype'] = adata.obs['leiden'].astype(str)+"_"+adata.obs['new_celltype'].astype(str)

In [ ]:
#check that healthy patients in the adata
adata.obs['patient_alias'].value_counts()

In [ ]:
#check that the timepoints are correct
adata.obs['timepoint_coarse'].value_counts()

In [ ]:
#check that the new_celltypes are correct
adata.obs['new_celltype'].value_counts()

In [ ]:
adata.obs['disease_state'].value_counts()

In [ ]:
adata[adata.obs['disease_state'] == "Healthy"].obs[['patient','leiden_and_celltype']].value_counts()

In [ ]:
#run infercnv with healthy as reference
cnv.tl.infercnv(
    adata,
    reference_key="disease_state",
    reference_cat=["Healthy"],
    window_size=250,
)

#### Disease state

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="disease_state")

In [ ]:

cnv_plot = cnv.pl.chromosome_heatmap_summary(adata[adata.obs['disease_state'] == "Healthy"], groupby="leiden_and_celltype",show=False)


In [ ]:
cnv.pl.chromosome_heatmap_summary(adata, groupby="leiden_and_celltype")

#### celltype

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="new_celltype")

#### patient by timepoint

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['timepoint_coarse'] == "Healthy")|(adata.obs['timepoint_coarse'] == "pre")], groupby="patient_alias")

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['timepoint_coarse'] == "Healthy")|(adata.obs['timepoint_coarse'] == "mid")], groupby="patient_alias")

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['timepoint_coarse'] == "Healthy")|(adata.obs['timepoint_coarse'] == "post")], groupby="patient_alias")

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['timepoint_coarse'] == "Healthy")|(adata.obs['timepoint_coarse'] == "progression")], groupby="patient_alias")

In [ ]:
cnv.tl.pca(adata) 
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata)
cnv.tl.umap(adata)
cnv.tl.cnv_score(adata)
sc.pl.umap(adata, color=["cnv_score"], ncols=1)

In [ ]:
adata

In [ ]:
#adata.write_h5ad("12_InferCNV.h5ad")

# Patient by patient CNV analysis

In [ ]:
#adata = sc.read_h5ad("12_InferCNV.h5ad")

In [ ]:
cnv.pl.umap(adata, color=["cnv_leiden"], legend_loc="on data")

In [ ]:
cnv.pl.umap(adata, color=["new_celltype"])

In [ ]:
cnv.pl.umap(adata, color=["patient_alias"])

In [ ]:
cnv.pl.umap(adata, color=["leiden"], legend_loc="on data")

In [ ]:
sc.pl.umap(adata, color="leiden", legend_loc="on data")

## P17

In [ ]:
cnv.pl.umap(adata[adata.obs['patient_alias'] == "P17"], color=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == "P17"], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P17") & (adata.obs["timepoint_coarse"] == "pre")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['patient_alias'] == "P17") & (adata.obs["timepoint_coarse"] == "pre")], groupby='new_celltype')

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P17") & (adata.obs["timepoint_coarse"] == "mid")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P17") & (adata.obs["timepoint_coarse"] == "progression")], groupby=['new_celltype'])

## P18

In [ ]:
cnv.pl.umap(adata[adata.obs['patient_alias'] == "P18"], color=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == "P18"], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P18") & (adata.obs["timepoint_coarse"] == "pre")], groupby=['leiden_and_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P18") & (adata.obs["timepoint_coarse"] == "mid")], groupby=['leiden_and_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P18") & (adata.obs["leiden"] == "9")], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P18") & (adata.obs["timepoint_coarse"] == "progression")], groupby=['leiden_and_celltype'])

## P12

In [ ]:
cnv.pl.umap(adata[adata.obs['patient_alias'] == "P12"], color=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == "P12"], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P12") & (adata.obs["timepoint_coarse"] == "pre")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P12") & (adata.obs["timepoint_coarse"] == "mid")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P12") & (adata.obs["timepoint_coarse"] == "post")], groupby=['new_celltype'])

## P09

In [ ]:
cnv.pl.umap(adata[adata.obs['patient_alias'] == "P09"], color=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == "P09"], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P09") & (adata.obs["timepoint_coarse"] == "pre")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P09") & (adata.obs["timepoint_coarse"] == "mid")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P09") & (adata.obs["timepoint_coarse"] == "post")], groupby=['new_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['disease_state'] == "MDS"], groupby='patient_alias', dendrogram=True)

In [ ]:
adata[adata.obs['patient_alias'] == "P11"].obs['timepoint_coarse'].value_counts()

#### P03

In [ ]:
adata

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == "P03"], groupby=['timepoint_coarse'])

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P03") & (adata.obs["timepoint_coarse"] == "pre")], groupby=['leiden_and_celltype'])

In [ ]:
p03 = adata[(adata.obs['patient_alias'] == "P03")].copy()
p03

In [ ]:
cnv.pl.chromosome_heatmap_summary(p03[p03.obs['timepoint_coarse'] == "pre"], groupby="leiden_and_celltype")

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P03") & (adata.obs["timepoint_coarse"] == "mid")], groupby=['leiden_and_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['patient_alias'] == "P03") & (adata.obs["timepoint_coarse"] == "mid")], groupby='leiden_and_celltype')

In [ ]:
cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == "P03") & (adata.obs["timepoint_coarse"] == "post")], groupby=['leiden_and_celltype'])

In [ ]:
cnv.pl.chromosome_heatmap_summary(adata[(adata.obs['patient_alias'] == "P03") & (adata.obs["timepoint_coarse"] == "post")], groupby='leiden_and_celltype')

#### Generate heatmaps for all patients

In [ ]:
for p in adata.obs['patient_alias'].unique():
    cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == p], 
    groupby=['timepoint_coarse'],
    save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_timepoints.png",
    show=False)
    plt.close()
    timepoints = adata[adata.obs['patient_alias'] == p].obs['timepoint_coarse'].unique()
    print(p, timepoints)
    for t in timepoints:
        cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == p) & (adata.obs["timepoint_coarse"] == t)], 
            groupby='leiden_and_celltype',
            save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_{t}.png",
            show=False)
        plt.close()

## Trajectories

In [ ]:
adata

In [ ]:
#pip install fa2-modified

In [ ]:
adata.obsm["X_cnv_2"] = adata.obsm["X_cnv"].toarray()

In [ ]:
sc.tl.dendrogram(adata, use_rep="X_cnv_2", groupby="leiden", key_added="cnv_leiden_dendrogram")

In [ ]:
sc.pl.dendrogram(adata, groupby='leiden', dendrogram_key="cnv_leiden_dendrogram", orientation="left")

In [ ]:
sc.tl.paga(adata, groups="leiden")


In [ ]:
sc.pl.paga(adata, layout="fr")

In [ ]:
# Get the value counts of the combinations of 'leiden' and 'new_celltype'
leiden_new_celltype_counts = adata.obs[['leiden', 'new_celltype']].value_counts()

# Create the dictionary with the Leiden value as prefix to the new_celltype
leiden_dict = {f"{leiden}": f"{leiden}_{new_celltype}" for (leiden, new_celltype), count in leiden_new_celltype_counts.items()}

# Print the resulting dictionary
print(leiden_dict)

In [ ]:
adata.obs["leiden_anno"] = adata.obs["leiden"].cat.rename_categories(leiden_dict)
adata.obs['leiden_anno']

In [ ]:
adata

### P18

In [ ]:
p18=adata[adata.obs['patient_alias'] == "P18"]

In [ ]:
sc.tl.paga(p18, groups="leiden_anno")

In [ ]:
p18.uns['paga_gex'] = p18.uns['paga'].copy()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib import patheffects

# Create the figure and axis
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Create the PAGA plot
sc.pl.paga(p18, layout="fr", fontsize=10, node_size_scale=5, title="PAGA Plot for P18", ax=ax, show=False)

# Change the color of the text within the plot (node labels) and add padding
for text in ax.texts:  # Loop through all text objects inside the plot
    text.set_color('magenta')  # Set text color to magenta
    text.set_fontsize(12)
    text.set_bbox(dict(facecolor='none', edgecolor='none', alpha=0.5, boxstyle='round,pad=0.4'))  # More padding
    

    # Outline the text with black
    text.set_path_effects([
        patheffects.withStroke(linewidth=1, foreground='black'),  # Add black outline
        patheffects.Normal()  # Set normal text effect
    ])

    # Adjust text positions if they are too close
    text.set_position((text.get_position()[0] + 0.03, text.get_position()[1] + 0.03))  # Adjust text position

# Change the title color to purple
plt.title("Gene expression PAGA Plot for P18", fontsize=20)

# Save the plot as PDF
plt.savefig("paga_plot_p18.pdf", dpi=300, bbox_inches='tight')

# Show the plot (optional)
plt.show()


In [ ]:
cnv.pp.neighbors(p18)
cnv.tl.umap(p18)

In [ ]:
sc.pp.neighbors(p18)
sc.tl.umap(p18)

In [ ]:
# Calculate paga based on CNV data
sc.tl.paga(p18, neighbors_key='cnv_neighbors', groups="leiden_anno")

In [ ]:
p18.uns['paga_cnv'] = p18.uns['paga'].copy()

In [ ]:
p18.uns['paga'] = p18.uns['paga_cnv']

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib import patheffects

# Create the figure and axis
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Create the PAGA plot
sc.pl.paga(p18, layout="fa", fontsize=10, node_size_scale=1, title="CNV PAGA Plot for P18", ax=ax, show=False)

# Change the color of the text within the plot (node labels) and add padding
for text in ax.texts:  # Loop through all text objects inside the plot
    text.set_color('magenta')  # Set text color to magenta
    text.set_fontsize(12)
    text.set_bbox(dict(facecolor='none', edgecolor='none', alpha=0.5, boxstyle='round,pad=0.4'))  # More padding
    

    # Outline the text with black
    text.set_path_effects([
        patheffects.withStroke(linewidth=1, foreground='black'),  # Add black outline
        patheffects.Normal()  # Set normal text effect
    ])

    # Adjust text positions if they are too close
    text.set_position((text.get_position()[0] + 0.03, text.get_position()[1] + 0.03))  # Adjust text position

# Change the title color to purple
plt.title("CNV PAGA Plot for P18", fontsize=20)

# Save the plot as PDF
plt.savefig("cnv_paga_plot_p18.pdf", dpi=300, bbox_inches='tight')

# Show the plot (optional)
plt.show()


In [ ]:
p18.obs['leiden_anno'].value_counts()

In [ ]:
p18_pruned = p18.obs['leiden_anno']

In [ ]:
adata.obs['patient_alias'].value_counts()

In [ ]:
#dendrogram based on CNV
sc.tl.dendrogram(p18, use_rep="X_cnv_2", groupby="leiden_anno", key_added="cnv_leiden_dendrogram")

In [ ]:
# Create the figure and axes for the subplot
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Plot dendrogram based on CNV in the specified subplot
sc.pl.dendrogram(p18, groupby='leiden_anno', dendrogram_key="cnv_leiden_dendrogram", ax=ax, show=False)

# Set the title for the dendrogram plot
ax.set_title("Dendrogram of P18 based on CNV", fontsize=10)

# Show the plot (optional)
plt.show()

In [ ]:
#dendrogram based on gex
sc.tl.dendrogram(p18, groupby="leiden_anno", key_added="gex_leiden_dendrogram")

In [ ]:
# Create the figure and axes for the subplot
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Plot dendrogram based on CNV in the specified subplot
sc.pl.dendrogram(p18, groupby='leiden_anno', dendrogram_key="gex_leiden_dendrogram", ax=ax, show=False)

# Set the title for the dendrogram plot
ax.set_title("Dendrogram of P18 based on gene expression", fontsize=10)

# Show the plot (optional)
plt.show()

In [ ]:
sc.pl.umap(p18, color='leiden', legend_loc='on data')

In [ ]:
sc.tl.umap(p18,init_pos='paga')

In [ ]:
sc.pl.umap(p18, color='leiden', legend_loc='on data')

In [ ]:
import numpy as np
p18.uns['iroot'] = np.flatnonzero(p18.obs['leiden_anno'] == '13_MEP')[0]

In [ ]:
sc.tl.dpt(p18)

In [ ]:
sc.pl.umap(p18, color=['dpt_pseudotime'], legend_loc='on data')

In [ ]:
cnv.pl.umap(p18, color='leiden', legend_loc='on data')

In [ ]:
p18

In [ ]:
import numpy as np
p18.uns['iroot'] = np.flatnonzero(p18.obs['leiden_anno'] == '13_MEP')[0]

In [ ]:
p18.obsm['X_cnv']

In [ ]:
p18.obs['gex_dpt_pseudotime'] = p18.obs['dpt_pseudotime'].copy()

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['disease_state'] == "Healthy"], groupby='leiden_anno')

In [ ]:
sc.pl.umap(adata, groups=['disease_state'])

In [ ]:
adata[adata.obs['disease_state'] == "Healthy"].obs['patient_alias'].value_counts()

In [ ]:
sc.pl.umap(adata, color='disease_state')

# InferCNV without patient C5

In [ ]:
adata_withC5 = adata.copy()

In [ ]:
adata_withC5

In [ ]:
adata_withC5.obs['disease_state'].value_counts()

In [ ]:
#adata = adata_withC5
adata = adata[adata.obs['patient_alias'] != "C5"]

In [ ]:
adata

In [ ]:
adata = adata[adata.obs['disease_state'] != "not available"]
adata

In [ ]:
adata.obs['patient_alias'].value_counts()

In [ ]:
#run infercnv with healthy as reference
cnv.tl.infercnv(
    adata,
    reference_key="disease_state",
    reference_cat=["Healthy"],
    window_size=250,
)

In [ ]:
#adata.write_h5ad("12_InferCNV_noC5.h5ad")

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby='disease_state')

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs['disease_state'] == "Healthy"], groupby='leiden_and_celltype')

In [ ]:
adata.uns['cnv']

In [ ]:
# Manually draw the heatmap
import matplotlib.axes
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from matplotlib.colors import Colormap, TwoSlopeNorm
from scanpy.plotting._utils import savefig_or_show
from scipy.sparse import issparse


tmp_adata = AnnData(X=adata.obsm['X_cnv'], obs=adata.obs, uns=adata.uns)
tmp_adata

In [ ]:
# re-sort, as saving & loading anndata destroys the order
chr_pos_dict = dict(sorted(adata.uns['cnv']["chr_pos"].items(), key=lambda x: x[1]))
chr_pos = list(chr_pos_dict.values())

# add chromosome annotations
var_group_positions = list(zip(chr_pos, chr_pos[1:] + [tmp_adata.shape[1]], strict=False))
var_group_positions


In [ ]:
sc.pl.heatmap(
        tmp_adata,
        var_names=tmp_adata.var.index.values,
        groupby="patient_alias",
        cmap='bwr',
        figsize=(16,10),
        show_gene_labels=False,
        var_group_positions=var_group_positions,
        var_group_labels=list(chr_pos_dict.keys()),
    )


In [ ]:
adata.obsm['X_cnv'].min()

In [ ]:
adata.obsm['X_cnv'].max()

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby='patient_alias', vmin=-0.2, vmax=0.4)

In [ ]:
for p in adata.obs['patient_alias'].unique():
    cnv.pl.chromosome_heatmap(adata[adata.obs['patient_alias'] == p], 
    groupby=['timepoint_coarse'],
    vmin=-0.2, 
    vmax=0.4,
    save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_timepoints_v2.png",
    show=False)
    plt.close()
    timepoints = adata[adata.obs['patient_alias'] == p].obs['timepoint_coarse'].unique()
    print(p, timepoints)
    for t in timepoints:
        cnv.pl.chromosome_heatmap(adata[(adata.obs['patient_alias'] == p) & (adata.obs["timepoint_coarse"] == t)], 
            groupby='leiden_and_celltype',
            vmin=-0.2, 
            vmax=0.4,
            save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_{t}_v2.png",
            show=False)
        plt.close()

In [ ]:
#run pca on the cnv stuff

cnv.tl.pca(adata)

In [ ]:
# P18 

p18 = adata[adata.obs['patient_alias'] == "P18"]

In [ ]:
#count cells per celltype

celltype_counts = p18.obs['leiden_and_celltype'].value_counts()
celltype_counts

In [ ]:
#Identify celltypes
valid_celltypes = celltype_counts[celltype_counts >= 37].index

#filter the celltype
p18_filtered = p18[p18.obs['leiden_and_celltype'].isin(valid_celltypes)].copy()

print(f"Original number of cells: {p18.n_obs}")
print(f"Filtered number of cells: {p18_filtered.n_obs}")

In [ ]:
cnv.tl.pca(p18_filtered)

In [ ]:
p18_filtered

In [ ]:
#dendrogram based on CNV
sc.tl.dendrogram(p18_filtered, use_rep="X_cnv_pca", groupby="leiden_anno", key_added="cnv_leiden_dendrogram")

In [ ]:
# Create the figure and axes for the subplot
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Plot dendrogram based on CNV in the specified subplot
sc.pl.dendrogram(p18_filtered, groupby='leiden_anno', dendrogram_key="cnv_leiden_dendrogram", ax=ax, show=False)

# Set the title for the dendrogram plot
ax.set_title("Dendrogram of P18 based on CNV", fontsize=10)

# Show the plot (optional)
plt.show()

In [ ]:
sc.tl.pca(p18_filtered)

In [ ]:
#dendrogram based on gex
sc.tl.dendrogram(p18_filtered, groupby="leiden_anno", key_added="gex_leiden_dendrogram")

In [ ]:
# Create the figure and axes for the subplot
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Plot dendrogram based on CNV in the specified subplot
sc.pl.dendrogram(p18_filtered, groupby='leiden_anno', dendrogram_key="gex_leiden_dendrogram", ax=ax, show=False)

# Set the title for the dendrogram plot
ax.set_title("Dendrogram of P18 based on gene expression", fontsize=10)

# Show the plot (optional)
plt.show()

# InferCNV without patient specific cluster cells

In [ ]:
adata[adata.obs['leiden_and_celltype'] == "4_Patient specific cluster"].obs['patient_alias'].value_counts()

In [ ]:
adata_filtered = adata[~((adata.obs['disease_state'] == "Healthy") & 
                         (adata.obs['new_celltype'] == "Patient specific cluster")) & 
                       ~(adata.obs['disease_state'] == "not available")].copy()

In [ ]:
adata_filtered.obs['patient_alias'].value_counts()

In [ ]:
adata_filtered.obs['disease_state'].value_counts() 

In [ ]:
#compare with original adata
adata.obs['disease_state'].value_counts()

In [ ]:
#run infercnv with healthy as reference
cnv.tl.infercnv(
    adata_filtered,
    reference_key="disease_state",
    reference_cat=["Healthy"],
    window_size=250,
)

In [ ]:
adata_filtered.write_h5ad("12_inferCNV_healthy_noPSC.h5ad")

In [ ]:
adata_filtered

In [ ]:
cnv.pl.chromosome_heatmap(adata_filtered[adata_filtered.obs['disease_state'] == "Healthy"], groupby='leiden_and_celltype')

In [ ]:
for p in adata_sc_mut.obs['patient_alias'].unique():
    cnv.pl.chromosome_heatmap(adata_sc_mut[adata_sc_mut.obs['patient_alias'] == p], 
    groupby=['timepoint_coarse'],
    vmin=-0.5, 
    vmax=0.5,
    save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_timepoints_v4.png",
    show=False)
    plt.close()
    timepoints = adata_sc_mut[adata_sc_mut.obs['patient_alias'] == p].obs['timepoint_coarse'].unique()
    print(p, timepoints)
    for t in timepoints:
        cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == p) & (adata_sc_mut.obs["timepoint_coarse"] == t)], 
            groupby='leiden_and_celltype',
            vmin=-0.5, 
            vmax=0.5,
            save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_{t}_v4.png",
            show=False)
        plt.close()

In [ ]:
adata_sc_mut[adata_sc_mut.obs['disease_state'] == "Healthy"]

In [ ]:
cnv.pl.chromosome_heatmap(adata_sc_mut[adata_sc_mut.obs['disease_state'] == "Healthy"], groupby='leiden_and_celltype')

In [ ]:
adata_filtered

In [ ]:
#add in the mutation data into the adata object

sc_mut = pd.read_csv("data/single_cell_mutation_for_priscilla.csv")
sc_mut = sc_mut.iloc[:,1:]
sc_mut.set_index("barcode_index", inplace=True)
sc_mut.fillna("na", inplace=True)
sc_mut


In [ ]:
sc_mut = sc_mut.drop(columns=["BARCODE"])

In [ ]:
#check if the ids are duplicated
adata_filtered.obs_names.duplicated().sum()

In [ ]:
missing_ids = sc_mut.index.difference(adata_filtered.obs.index)
if not missing_ids.empty:
    print("Warning: The following IDs are in df but not in adata.obs:", missing_ids)

# Proceed with reindexing or merging
sc_mut = sc_mut.reindex(adata_filtered.obs.index).fillna("na")
adata_sc_mut = adata_filtered.copy()
adata_sc_mut.obs = adata_sc_mut.obs.join(sc_mut)
adata_sc_mut.obs

In [ ]:
adata_sc_mut.obs[['timepoint','timepoint_coarse']].value_counts()

In [ ]:
print(adata_sc_mut.obs.dtypes)

In [ ]:
adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns] = adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns].astype(str).astype("category")

In [ ]:
#adata_sc_mut.write_h5ad("12_infercnv_nohealthyPCS_with_mutation_data")

# Last edited

In [ ]:
adata_sc_mut = sc.read_h5ad("12_infercnv_nohealthyPCS_with_mutation_data.h5ad")

In [ ]:
# check the healthy and disease distribution
sc.pl.umap(adata_sc_mut, color='disease_state', palette={"Healthy":"black","MDS":"#E0B0FF"})

In [ ]:
sc.pl.umap(adata_sc_mut, color="leiden", legend_loc="on data")

In [ ]:
adata_sc_mut.obs.loc[adata_sc_mut.obs["leiden"] == "13", "new_celltype"] = "Patient specific cluster"

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata_sc_mut, color="new_celltype", show=False, ax=ax)
ax.set_rasterization_zorder(0)
plt.savefig("figures/new_celltype_umap_v2.pdf", dpi=300, bbox_inches='tight', format="pdf")

In [ ]:
# Create new obs for infercnv celltype labels

adata_sc_mut.obs["infercnv_new_celltype"] = adata_sc_mut.obs["new_celltype"]

adata_sc_mut.obs["infercnv_new_celltype"] = adata_sc_mut.obs["infercnv_new_celltype"].cat.add_categories([
    "P18 specific cluster A", "P18 specific cluster B", "P18 specific cluster C"
])

adata_sc_mut.obs.loc[adata_sc_mut.obs["leiden"] == "13", "infercnv_new_celltype"] = "P18 specific cluster A"
adata_sc_mut.obs.loc[adata_sc_mut.obs["leiden"] == "9", "infercnv_new_celltype"] = "P18 specific cluster B"
adata_sc_mut.obs.loc[adata_sc_mut.obs["leiden"] == "15", "infercnv_new_celltype"] = "P18 specific cluster C"

In [ ]:
adata_sc_mut.obs['infercnv_new_celltype']

In [ ]:
adata_sc_mut.uns['infercnv_new_celltype_colors']

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata_sc_mut, color="infercnv_new_celltype", palette=['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#8c564b',
 '#e377c2',
 '#b5bd61',
 '#17becf',
 '#1f77b4',
 '#1f77b4',
 '#1f77b4'], ax=ax, show=False)

# Manually rasterize all scatter collections
for coll in ax.collections:
    coll.set_rasterized(True)

fig.savefig("figures/infercnv_new_celltype_umap.pdf", dpi=300, bbox_inches='tight', format="pdf")

In [ ]:
muts = ['chr17_76736877_G_A_call',
'chr17_76736877_G_C_call',
'chr17_76736877_G_T_call',
'chr17_7674250_C_T_call', 
'chr17_7675082_G_T_call',
'chr17_7676051_G_C_call']

In [ ]:
adata_sc_mut.obs['patient_alias'].unique()

In [ ]:
for p in adata_sc_mut.obs['patient_alias'].unique():
    for m in muts:
            cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == p)],
            groupby=["timepoint_coarse",m],
            vmin=-0.5,
            vmax=0.5,
            save=f"figures/infercnv_heatmaps/cnv_heatmap_{p}_{m}.png",
            show=False)
            plt.close()

In [ ]:
adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "C1") & (adata_sc_mut.obs["chr17_7676051_G_C_call"] == "wt")].obs["dataset"]

In [ ]:
adata_sc_mut[adata_sc_mut.obs_names == "TTTATGCAGCTGTTCA-1"]

In [ ]:
adata_sc_mut[adata_sc_mut.obs['patient_alias'] == "P18"].obs["chr17_7674250_C_T_call"].value_counts()

In [ ]:
cnv_plot = cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "P18") & (adata_sc_mut.obs["chr17_7674250_C_T_call"] == "wt")], 
groupby=["timepoint_coarse","infercnv_new_celltype","chr17_7674250_C_T_call"], vmin=-0.5, vmax=0.5, show=False),#save="figures/infercnv_heatmaps/cnv_heatmap_P18_timepoint_celltype_chr17_7674250_C_T_call_wt.png")

cnv_plot_dict = cnv_plot[0]

# Get the axes
ax = cnv_plot_dict["heatmap_ax"]

In [ ]:
adata_sc_mut[
    (adata_sc_mut.obs['patient_alias'] == "P18") & 
    (adata_sc_mut.obs["chr17_7674250_C_T_call"] == "wt") & 
    (adata_sc_mut.obs["new_celltype"] == "Patient specific cluster")].obs[
    ['BARCODE','dataset','timepoint_coarse','leiden','infercnv_new_celltype','chr17_7674250_C_T_call']]

In [ ]:
# Get figure from one of the axes
fig = ax.get_figure()

# Use .obs_names as a proxy for cell order (assuming no reordering in the plot function)
cell_order = adata_sc_mut.obs_names.tolist()

# Wrap in expected structure
cnv_plot_fixed = {
    "fig": fig,
    "heatmap_ax": ax,
    "cell_order": cell_order  # or replace with real order if you sorted
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

def add_group_colorbars_to_cnv_heatmap(adata, cnv_plot, 
                                       timepoint_col="timepoint", 
                                       celltype_col="new_celltype", 
                                       palette1="Set2", 
                                       palette2="tab20"):
    """
    Add two vertical colorbars to a CNV heatmap showing timepoint and celltype.

    Parameters:
    - adata: AnnData object used to generate the CNV heatmap.
    - cnv_plot: dictionary returned from cnv.pl.chromosome_heatmap_summary(show=False).
    - timepoint_col: obs column name for timepoint grouping.
    - celltype_col: obs column name for celltype grouping.
    - palette1: seaborn palette for timepoint.
    - palette2: seaborn palette for celltype.
    """

    fig = cnv_plot["fig"]
    ax = cnv_plot["heatmap_ax"]
    cell_order = cnv_plot["cell_order"]  # Ensures correct row alignment

    # Subset obs in correct order
    obs = adata.obs.loc[cell_order]

    # Create color maps
    tp_vals = obs[timepoint_col].unique()
    ct_vals = obs[celltype_col].unique()
    tp_colors = dict(zip(tp_vals, sns.color_palette(palette1, len(tp_vals))))
    ct_colors = dict(zip(ct_vals, sns.color_palette(palette2, len(ct_vals))))

    # Map to color arrays
    tp_array = np.array(obs[timepoint_col].map(tp_colors).tolist()).reshape(-1, 1, 3)
    ct_array = np.array(obs[celltype_col].map(ct_colors).tolist()).reshape(-1, 1, 3)

    # Make space for colorbars
    fig.subplots_adjust(left=0.22)

    # Add new axes for colorbars
    y0 = ax.get_position().y0
    height = ax.get_position().height
    ax_tp = fig.add_axes([0.18, y0, 0.01, height])
    ax_ct = fig.add_axes([0.20, y0, 0.01, height])

    ax_tp.imshow(tp_array, aspect='auto')
    ax_ct.imshow(ct_array, aspect='auto')

    ax_tp.axis('off')
    ax_ct.axis('off')

    # Add legends (optional)
    tp_legend = [Patch(color=color, label=label) for label, color in tp_colors.items()]
    ct_legend = [Patch(color=color, label=label) for label, color in ct_colors.items()]

    fig.legend(handles=tp_legend, title='Timepoint', loc='upper left', bbox_to_anchor=(0.005, 0.99))
    fig.legend(handles=ct_legend, title='Celltype', loc='lower left', bbox_to_anchor=(0.005, 0.01))

    plt.draw()


In [ ]:
add_group_colorbars_to_cnv_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "P18") & (adata_sc_mut.obs["chr17_7674250_C_T_call"] == "wt")],
cnv_plot_fixed, timepoint_col="timepoint_coarse", celltype_col="infercnv_new_celltype")

In [ ]:
cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "P18") & (adata_sc_mut.obs["chr17_7674250_C_T_call"] == "mt")], 
groupby=["timepoint_coarse","infercnv_new_celltype","chr17_7674250_C_T_call"], vmin=-0.5, vmax=0.5,figsize=(10,25),
save="figures/infercnv_heatmaps/cnv_heatmap_P18_timepoint_celltype_chr17_7674250_C_T_call_mt.png")

In [ ]:
cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "P03") & (adata_sc_mut.obs["chr17_7675082_G_T_call"] == "wt")], 
groupby=["timepoint_coarse","infercnv_new_celltype","chr17_7675082_G_T_call"], 
vmin=-0.5, 
vmax=0.5,
figsize=(10,25),
save="figures/infercnv_heatmaps/cnv_heatmap_P03_timepoint_celltype_chr17_7675082_G_T_call_wt.png")


In [ ]:
cnv.pl.chromosome_heatmap(adata_sc_mut[(adata_sc_mut.obs['patient_alias'] == "P03") & (adata_sc_mut.obs["chr17_7675082_G_T_call"] == "mt")], 
groupby=["timepoint_coarse","infercnv_new_celltype","chr17_7675082_G_T_call"], 
vmin=-0.5, 
vmax=0.5,
figsize=(10,25),
save="figures/infercnv_heatmaps/cnv_heatmap_P03_timepoint_celltype_chr17_7675082_G_T_call_mt.png")


In [ ]:
#adata_sc_mut.write_h5ad("12_infercnv_nohealthyPCS_with_mutation_data_updated_annotations.h5ad")

In [ ]:
p18 = adata_sc_mut[adata_sc_mut.obs['patient_alias'] == "P18"]

In [ ]:
p18.obs['infercnv_new_celltype']

In [ ]:
p18.obs["timepoint_coarse_new_celltype"] = p18.obs["timepoint_coarse"].astype(str) + "_" + p18.obs["infercnv_new_celltype"].astype(str)


## Updating annotation and recreating the InferCNV heatmap

In [ ]:
adata = sc.read_h5ad("17_updated_annotations_infercnv_mutations_scores.h5ad")

In [ ]:
adata

### Updating annotations

In [ ]:
#Define a new celltype obs
adata.obs["celltype_v2"] = adata.obs["infercnv_new_celltype"]

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc="on data")

In [ ]:
adata.obs[["leiden", "celltype_v2"]].value_counts()

In [ ]:
anno_map ={"13":"Atypical cluster A",
"9":"Atypical cluster B",
"15":"Atypical cluster C",
"8":"Atypical cluster D",
"20":"Atypical cluster E",
"22":"Atypical cluster F",
"18":"Atypical cluster G",
"7":"Atypical cluster H",
"10":"Atypical cluster I",
"3":"Atypical cluster J",
"19":"Atypical cluster K",
"0":"Atypical cluster L",
"4":"Atypical cluster M",
"14":"Atypical cluster N",
"25":"Atypical cluster O"}

In [ ]:
#map the leiden cluster from the anno_map and replace the celltype_v2
adata.obs["celltype_v2"] = adata.obs["leiden"].map(anno_map).combine_first(adata.obs["infercnv_new_celltype"])
adata.obs[["leiden", "celltype_v2"]].value_counts()


In [ ]:
ct_list = adata.obs["celltype_v2"].cat.categories.tolist()
ct_list

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(adata, color="celltype_v2", show=False, ax=ax,legend_loc="on data", groups=['Atypical cluster A',
 'Atypical cluster B',
 'Atypical cluster C',
 'Atypical cluster D',
 'Atypical cluster E',
 'Atypical cluster F',
 'Atypical cluster G',
 'Atypical cluster H',
 'Atypical cluster I',
 'Atypical cluster J',
 'Atypical cluster K',
 'Atypical cluster L',
 'Atypical cluster M',
 'Atypical cluster N',
 'BMCP',
 'Erythroblast',
 'GMP',
 'HSC',
 'MEP',
 'MEP/MKP',
 'MPP',
 'Monocyte progenitor'], na_in_legend=False, legend_fontsize=5)
#plt.savefig("figures/celltype_v2_umap_legend_onloc.pdf", dpi=300, bbox_inches='tight', format="pdf")

### Update patient alias colours

In [ ]:
pa_color_df = pd.read_csv("data/2025_04_23_colors_for_priscilla_v2.csv")
pa_color_df

In [ ]:
#convert dataframe to dictionary
pa_color_dict_new = pa_color_df.set_index('Id')['hex'].to_dict()
pa_color_dict_new

In [ ]:
adata.obs['patient_alias'].cat.categories

In [ ]:
adata.uns['patient_alias_colors']

In [ ]:
#make a dictionary from adata.obs['patient_alias'].cat.categories and adata.uns['patient_alias_colors']
pa_color_dict_old = dict(zip(adata.obs['patient_alias'].cat.categories, adata.uns['patient_alias_colors']))

#replace colors in pa_color_dict_old with colors in pa_color_dict_new
pa_color_dict = {k: pa_color_dict_new.get(k, v) for k, v in pa_color_dict_old.items()}
pa_color_dict

In [ ]:
sc.pl.umap(adata, 
color="patient_alias",
palette=pa_color_dict)

In [ ]:
adata.write_h5ad("05052025_adata_celltypev2_updated_colors.h5ad")

### InferCNV heatmap recreation

In [ ]:
import matplotlib.axes
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from matplotlib.colors import Colormap, TwoSlopeNorm
from scanpy.plotting._utils import savefig_or_show
from scipy.sparse import issparse


tmp_adata = AnnData(X=adata.obsm[f"X_cnv"], obs=adata.obs, uns=adata.uns)

# re-sort, as saving & loading anndata destroys the order
chr_pos_dict = dict(sorted(adata.uns['cnv']["chr_pos"].items(), key=lambda x: x[1]))
chr_pos = list(chr_pos_dict.values())

# center color map at 0
tmp_data = tmp_adata.X.data if issparse(tmp_adata.X) else tmp_adata.X
#vmin = kwargs.pop("vmin", None)
#vmax = kwargs.pop("vmax", None)
#if vmin is None:
vmin = np.nanmin(tmp_data)
#if vmax is None:
vmax = np.nanmax(tmp_data)
#kwargs["norm"] = TwoSlopeNorm(0, vmin=vmin, vmax=vmax)
norm = TwoSlopeNorm(0, vmin=vmin, vmax=vmax)

var_group_positions = list(zip(chr_pos, chr_pos[1:] + [tmp_adata.shape[1]], strict=False))


In [ ]:
from collections.abc import Collection, Mapping, Sequence
from scanpy._utils import sanitize_anndata, _check_use_raw
from scanpy import get
adata.obsm['X_cnv']
def _prepare_dataframe(
    adata: AnnData,
    var_names: _VarNames | Mapping[str, _VarNames],
    groupby: str | Sequence[str] | None = None,
    *,
    use_raw: bool | None = None,
    log: bool = False,
    num_categories: int = 7,
    layer: str | None = None,
    gene_symbols: str | None = None,
) -> tuple[Sequence[str], pd.DataFrame]:
    """
    Given the anndata object, prepares a data frame in which the row index are the categories
    defined by group by and the columns correspond to var_names.

    Parameters
    ----------
    adata
        Annotated data matrix.
    var_names
        `var_names` should be a valid subset of  `adata.var_names`.
    groupby
        The key of the observation grouping to consider. It is expected that
        groupby is a categorical. If groupby is not a categorical observation,
        it would be subdivided into `num_categories`.
    use_raw
        Whether to use `raw` attribute of `adata`. Defaults to `True` if `.raw` is present.
    log
        Use the log of the values.
    layer
        AnnData layer to use. Takes precedence over `use_raw`
    num_categories
        Only used if groupby observation is not categorical. This value
        determines the number of groups into which the groupby observation
        should be subdivided.
    gene_symbols
        Key for field in .var that stores gene symbols.

    Returns
    -------
    Tuple of `pandas.DataFrame` and list of categories.
    """

    sanitize_anndata(adata)
    use_raw = _check_use_raw(adata, use_raw, layer=layer)
    if isinstance(var_names, str):
        var_names = [var_names]

    groupby_index = None
    if groupby is not None:
        if isinstance(groupby, str):
            # if not a list, turn into a list
            groupby = [groupby]
        for group in groupby:
            if group not in list(adata.obs_keys()) + [adata.obs.index.name]:
                if adata.obs.index.name is not None:
                    msg = f' or index name "{adata.obs.index.name}"'
                else:
                    msg = ""
                raise ValueError(
                    "groupby has to be a valid observation. "
                    f"Given {group}, is not in observations: {adata.obs_keys()}" + msg
                )
            if group in adata.obs.columns and group == adata.obs.index.name:
                msg = (
                    f"Given group {group} is both and index and a column level, "
                    "which is ambiguous."
                )
                raise ValueError(msg)
            if group == adata.obs.index.name:
                groupby_index = group
    if groupby_index is not None:
        # obs_tidy contains adata.obs.index
        # and does not need to be given
        groupby = groupby.copy()  # copy to not modify user passed parameter
        groupby.remove(groupby_index)
    keys = list(groupby) + list(np.unique(var_names))
    obs_tidy = get.obs_df(
        adata, keys=keys, layer=layer, use_raw=use_raw, gene_symbols=gene_symbols
    )
    assert np.all(np.array(keys) == np.array(obs_tidy.columns))

    if groupby_index is not None:
        # reset index to treat all columns the same way.
        obs_tidy.reset_index(inplace=True)
        groupby.append(groupby_index)

    if groupby is None:
        categorical = pd.Series(np.repeat("", len(obs_tidy))).astype("category")
    elif len(groupby) == 1 and is_numeric_dtype(obs_tidy[groupby[0]]):
        # if the groupby column is not categorical, turn it into one
        # by subdividing into  `num_categories` categories
        categorical = pd.cut(obs_tidy[groupby[0]], num_categories)
    elif len(groupby) == 1:
        categorical = obs_tidy[groupby[0]].astype("category")
        categorical.name = groupby[0]
    else:
        # join the groupby values  using "_" to make a new 'category'
        categorical = obs_tidy[groupby].apply("_".join, axis=1).astype("category")
        categorical.name = "_".join(groupby)

        # preserve category order
        from itertools import product

        order = {
            "_".join(k): idx
            for idx, k in enumerate(
                product(*(obs_tidy[g].cat.categories for g in groupby))
            )
        }
        categorical = categorical.cat.reorder_categories(
            sorted(categorical.cat.categories, key=lambda x: order[x])
        )
    obs_tidy = obs_tidy[var_names].set_index(categorical)
    categories = obs_tidy.index.categories

    if log:
        obs_tidy = np.log1p(obs_tidy)

    return categories, obs_tidy

In [ ]:
#from scanpy.plotting._anndata import _prepare_dataframe

categories, obs_tidy = _prepare_dataframe(
    tmp_adata,
    var_names=tmp_adata.var.index.values,
    groupby=["patient_alias","timepoint_coarse"],
    use_raw=None,
    log=None,
    num_categories=None,  # TODO: fix this line
    gene_symbols=None,
    layer=None,
)

In [ ]:
obs_tidy

In [ ]:
tmp_adata

In [ ]:
mut_call = ['chr17_76736877_G_A_call', 
'chr17_76736877_G_C_call', 
'chr17_76736877_G_T_call', 
'chr17_7674250_C_T_call', 
'chr17_7675082_G_T_call', 
'chr17_7676051_G_C_call']

In [ ]:
# Define your mutation columns
srsf2_cols = ['chr17_76736877_G_A_call', 'chr17_76736877_G_C_call', 'chr17_76736877_G_T_call']
tp53_cols = ['chr17_7674250_C_T_call', 'chr17_7675082_G_T_call', 'chr17_7676051_G_C_call']

mut_df =  tmp_adata.obs[mut_call]

# Mapping dictionary
mapping = {"mt": 1, "wt": 0, "unknown": 0}

# Step 1: Binarize mutation columns
for col in srsf2_cols + tp53_cols:
    mut_df[col] = mut_df[col].map(mapping).fillna(0).astype(int)

# Step 2: Determine if a row is a TP53 mutant or SRSF2 mutant
mut_df["TP53_mutant"] = mut_df[tp53_cols].any(axis=1).astype(int)
mut_df["SRSF2_mutant"] = mut_df[srsf2_cols].any(axis=1).astype(int)

# Step 3: Classify mutant type
def classify_mutant(row):
    if row["TP53_mutant"] == 1 and row["SRSF2_mutant"] == 1:
        return "double"
    elif row["TP53_mutant"] == 1:
        return "TP53"
    elif row["SRSF2_mutant"] == 1:
        return "SRSF2"
    else:
        return "wildtype"

mut_df["mutant_type"] = mut_df.apply(classify_mutant, axis=1)
mut_df


In [ ]:
obs_tidy=tmp_adata.to_df()

In [ ]:
obs_tidy["patient_alias"] = tmp_adata.obs["patient_alias"]
obs_tidy["timepoint_coarse"] = tmp_adata.obs["timepoint_coarse"]
obs_tidy["celltype"] = tmp_adata.obs["infercnv_new_celltype"]
obs_tidy["mutant_type"] = mut_df["mutant_type"]
obs_tidy

In [ ]:
var_group_labels=list(chr_pos_dict.keys())

In [ ]:
groupby_colors = adata.uns["patient_alias_colors"]

In [ ]:
obs_tidy = obs_tidy.sort_index()

In [ ]:
from scanpy.plotting._utils import check_colornorm

colorbar_width = 0.2

In [ ]:
var_names=tmp_adata.var.index.values
var_names

In [ ]:
value_sum = 0
ticks = []  # list of centered position of the labels
labels = []
label2code = {}  # dictionary of numerical values asigned to each label
for code, (label, value) in enumerate(
    obs_tidy.index.value_counts(sort=False).items()
):
    ticks.append(value_sum + (value / 2))
    labels.append(label)
    value_sum += value
    label2code[label] = code

In [ ]:
len(np.array([[label2code[lab] for lab in obs_tidy.index]]).T)

In [ ]:
fig = plt.figure()
heatmap_ax = fig.add_subplot()
heatmap_ax.imshow(obs_tidy.values, aspect="auto", norm=norm, interpolation="nearest", cmap='bwr')
plt.show()

In [ ]:
from matplotlib import gridspec, patheffects, rcParams
from scanpy.plotting._anndata import _plot_colorbar, _plot_categories_as_colorblocks, _plot_gene_groups_brackets

# define a layout of 2 rows x 4 columns
# first row is for 'brackets' (if no brackets needed, the height of this row
# is zero) second row is for main content. This second row is divided into
# three axes:
#   first ax is for the categories defined by `groupby`
#   second ax is for the heatmap
#   third ax is for the dendrogram
#   fourth ax is for colorbar
dendrogram = None
figsize = None
show_gene_labels = False

dendro_width = 1 if dendrogram else 0
groupby_width = 0.2 if categorical else 0
if figsize is None:
    height = 6
    heatmap_width = len(var_names) * 0.3 if show_gene_labels else 8
    width = heatmap_width + dendro_width + groupby_width
else:
    width, height = figsize
    heatmap_width = width - (dendro_width + groupby_width)

if var_group_positions is not None and len(var_group_positions) > 0:
    # add some space in case 'brackets' want to be plotted on top of the image
    height_ratios = [0.15, height]
else:
    height_ratios = [0, height]

width_ratios = [
    groupby_width,
    heatmap_width,
    dendro_width,
    colorbar_width,
]
fig = plt.figure(figsize=(width, height))

axs = gridspec.GridSpec(
    nrows=2,
    ncols=4,
    width_ratios=width_ratios,
    wspace=0.15 / width,
    hspace=0.13 / height,
    height_ratios=height_ratios,
)

heatmap_ax = fig.add_subplot(axs[1, 1])
#kwds.setdefault("interpolation", "nearest")
im = heatmap_ax.imshow(obs_tidy.values, aspect="auto", norm=norm, interpolation="nearest", cmap='bwr')

heatmap_ax.set_ylim(obs_tidy.shape[0] - 0.5, -0.5)
heatmap_ax.set_xlim(-0.5, obs_tidy.shape[1] - 0.5)
heatmap_ax.tick_params(axis="y", left=False, labelleft=False)
heatmap_ax.set_ylabel("")
heatmap_ax.grid(visible=False)

if show_gene_labels:
    heatmap_ax.tick_params(axis="x", labelsize="small")
    heatmap_ax.set_xticks(np.arange(len(var_names)))
    heatmap_ax.set_xticklabels(var_names, rotation=90)
else:
    heatmap_ax.tick_params(axis="x", labelbottom=False, bottom=False)
# plot colorbar
_plot_colorbar(im, fig, axs[1, 3])


groupby_ax = fig.add_subplot(axs[1, 0])
(
    label2code,
    ticks,
    labels,
    groupby_cmap,
    norm,
) = _plot_categories_as_colorblocks(
    groupby_ax, obs_tidy, colors=groupby_colors, orientation="left"
)

# add lines to main heatmap
line_positions = (
    np.cumsum(obs_tidy.index.value_counts(sort=False))[:-1] - 0.5
)
heatmap_ax.hlines(
    line_positions,
    -0.5,
    len(var_names) - 0.5,
    lw=1,
    color="black",
    zorder=10,
    clip_on=False,
)

if dendrogram:
    dendro_ax = fig.add_subplot(axs[1, 2], sharey=heatmap_ax)
    _plot_dendrogram(
        dendro_ax, adata, groupby, dendrogram_key=_dk(dendrogram), ticks=ticks
    )

# plot group legends on top of heatmap_ax (if given)
if var_group_positions is not None and len(var_group_positions) > 0:
    gene_groups_ax = fig.add_subplot(axs[0, 1], sharex=heatmap_ax)
    _plot_gene_groups_brackets(
        gene_groups_ax,
        group_positions=var_group_positions,
        group_labels=var_group_labels,
        rotation=90,
        left_adjustment=-0.3,
        right_adjustment=0.3,
    )
